In [ ]:
class STNBlock(nn.Module):
    """
    Spatial Transformer Network (STN) block for spatial attention.
    Applies an affine transformation to the input feature map for better spatial focus.
    """
    def __init__(self, in_channels):
        super(STNBlock, self).__init__()

        # Localization network to predict the affine parameters
        self.localization = nn.Sequential(
            nn.AdaptiveAvgPool2d(7),
            nn.Conv2d(in_channels, 32, kernel_size=7, stride=1, padding=0),
            nn.ReLU(True),
            nn.Flatten(),
            nn.Linear(32 * 1 * 1, 128),
            nn.ReLU(True),
            nn.Linear(128, 6)
        )

        # Initialize weights and biases to produce an identity transformation initially
        self.localization[-1].weight.data.zero_()
        self.localization[-1].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

    def forward(self, x):
        # Predict affine transformation parameters
        theta = self.localization(x).view(-1, 2, 3)

        # Generate a grid of coordinates using the affine transformation
        grid = F.affine_grid(theta, x.size(), align_corners=False)

        # Apply the transformation to the input
        x = F.grid_sample(x, grid, align_corners=False)
        return x

